# Notes on GLORYS12 regridding

This notebook contains notes on how to properly regrid GLORYS12 to a coarser grid, quirks and fixes.

In [ ]:
import dask.distributed as distributed
import hvplot.xarray  # noqa
import numpy as np
import xarray as xr

from arcomake.processing import gaussian_blur_extrapolate, get_sea_mask, regrid

In [ ]:
local_cluster = distributed.LocalCluster(processes=False)
local_cluster

## Regridding the bathymetry

The first point to investigate is how to regrid the bathymetry, so to compute a new sea mask. The bathymetry is invalid on land points, and the `xarray-regrid` package provides algorithms to deal with nans, but each returns different results. Here we investigate which one should be used.

In [ ]:
bathymetry = xr.open_dataset("cm://cmems_mod_glo_phy_my_0.083deg_static", engine="copernicusmarine", dataset_part="bathy")
# The bathymetry includes points below -77 deg of latitude, which are missing/invalid in other variables
# Moreover, at 90 deg of latitude its nan
bathymetry = bathymetry.sel(latitude=slice(-76.9167, 89.9167))
bathymetry = bathymetry.compute()
bathymetry

In [ ]:
def cell_areas(ds, res_deg=0.25):
    """Get the area of a cell using a very crude approximation"""
    # Earth polar radius
    EARTH_RADIUS = 6_356_800.0
    latitudes = ds["latitude"].values
    longitudes = ds["longitude"].values
    latitudes_mesh, _ = np.meshgrid(latitudes, longitudes, indexing="ij")
    unit_radius_linearized_arc_length = 2 * np.tan((np.pi / 180) * res_deg / 2)
    e1t = xr.DataArray(data=EARTH_RADIUS * np.cos(latitudes_mesh * np.pi / 180.0) * unit_radius_linearized_arc_length, 
                       dims=("latitude", "longitude"), 
                       coords={"latitude": ds["latitude"], "longitude": ds["longitude"]})
    e2t = xr.DataArray(data=EARTH_RADIUS * np.full(shape=latitudes_mesh.shape, fill_value=unit_radius_linearized_arc_length), 
                       dims=("latitude", "longitude"), 
                       coords={"latitude": ds["latitude"], "longitude": ds["longitude"]})
    return e1t * e2t 

In [ ]:
coordinates = xr.open_dataset("cm://cmems_mod_glo_phy_my_0.083deg_static", engine="copernicusmarine", dataset_part="coords", variables=["e1t", "e2t"])
coordinates = coordinates.sel(latitude=slice(-76.9167, 89.9167))
# IMPORTANT! Notice that coordinates and bathy have mismatching coordinates due to approximation errorsd
coordinates = coordinates.assign_coords(coords=bathymetry.coords)
coordinates = coordinates.compute()
coordinates

In [ ]:
# At the north pole singularity point, the coordinates and bathymetry datasets contains nans
glorys_cell_areas = (coordinates["e1t"] * coordinates["e2t"])
computed_cell_areas = cell_areas(bathymetry, res_deg=1/12)

# The two estimates are not far away, their difference is within 1%
xr.testing.assert_allclose(glorys_cell_areas, computed_cell_areas, rtol=1e-2)

In [ ]:
 # Total volume of water is not far from the literature one, e.g. 1_335_000_000 cubic kilometers (see: https://oceanservice.noaa.gov/facts/oceanwater.html)
(bathymetry["deptho"] * glorys_cell_areas).sum()

In [ ]:
# Using our crude approximation we get similar numbers
(bathymetry["deptho"] * computed_cell_areas).sum()

In [ ]:
def regrid_0p25(ds, fill=False, method="conservative", **kwargs):
    """Simple arcomake regrid dataset to regrid to 1/4 deg of resolution"""
    grid = dict(south=-90.0, north=90.0, west=0.0, east=359.75, resolution_lat=0.25, resolution_lon=0.25)
    if fill:
        ds = ds.fillna(0.0)
    # regrid method and reduction method have the same keyword argument name
    if "reduce" in kwargs:
        reduce_method = kwargs.pop("reduce")
        kwargs["method"] = reduce_method
    ds_regridded = regrid(ds, grid=grid, method=method, kwargs=kwargs)
    return ds_regridded

The relevant piece of code from `xarray-regrid` that does conservative regridding is close to the following:

```python

def get_valid_threshold(nan_threshold: float) -> float:
    valid_threshold: float = 1 - np.clip(nan_threshold, 1e-6, 1.0 - 1e-6)
    return valid_threshold

if skipna:
    valid_frac = xr.dot(
        da.notnull(), *weight_arrays, dim=list(weights.keys()), optimize=True
    )

da_regrid: xr.DataArray = xr.dot(
    da.fillna(0), *weight_arrays, dim=list(weights.keys()), optimize=True
)

if skipna:
    da_regrid /= valid_frac
    da_regrid = da_regrid.where(valid_frac >= get_valid_threshold(nan_threshold))
```


Hence, prefilling with 0.0 when using `skipna=False` in `regrid_0p25` should return the same result.

In [ ]:
bathymetry_0p25 = regrid_0p25(bathymetry, fill=True, skipna=False)
bathymetry_0p25_nofill = regrid_0p25(bathymetry, fill=False, skipna=False)
xr.testing.assert_identical(bathymetry_0p25, bathymetry_0p25_nofill)

Setting the depth of land points to 0.0 and applying conservative regridding get consistent results, as far as total volume of water is concerned

In [ ]:
(cell_areas(bathymetry_0p25, res_deg=0.25) * bathymetry_0p25["deptho"]).sum()

By playing with `nan_threshold`, one gets bathymetries with will be different around coaslines, with values rescaled by `valid_frac`. Hence, the total water volume can be slighly higher or lower.

In [ ]:
def regridded_total_volume(fill, skipna, nan_threshold=0.0):
    bathymetry_regridded = regrid_0p25(bathymetry, fill=fill, skipna=skipna, nan_threshold=nan_threshold)
    bathymetry_regridded = bathymetry_regridded.fillna(0.0)
    total_volume = (cell_areas(bathymetry_regridded, res_deg=0.25) * bathymetry_regridded["deptho"]).sum()
    return total_volume.values.item()

In [ ]:
# Drop all target grid cells overlapping with one or more masked (nan) source grid cells.
# The result is less than previously computed total volume of water.
regridded_total_volume(fill=False, skipna=True, nan_threshold=0.0)

In [ ]:
# Keep target grid cells overlapping only with valid (not nan) source grid cells.
# The result is higher than previously computed total volume of water.
regridded_total_volume(fill=False, skipna=True, nan_threshold=1.0)

In [ ]:
# Drop all target grid cells whose area overlapping masked source cells is greater than half of their area.
# The result sits in between the two previously computed total volumes of water.
regridded_total_volume(fill=False, skipna=True, nan_threshold=0.5)

Notice that:

  1. As `get_valid_threshold` clips the `nan_threshold` value, grid cells with very small `valid_frac` can appear into target as valid data (and grid cells with `valid_frac` close to 1.0 may be filtered out).
  2. As `da_regrid` is rescaled by `valid_frac <= 1.0`, the target will contain greater depths than those computed with `skipna=False`.

In [ ]:
bathymetry_0p25_skipna = regrid_0p25(bathymetry, fill=False, skipna=True, nan_threshold=1.0)

(bathymetry_0p25_skipna >= bathymetry_0p25).where(bathymetry_0p25_skipna.notnull()).all() 

Both conceptually and numerically, the right method to regrid the bathymetry is using a conservative algorithm and `skipna=False`.

## Computing the sea mask

The GLORYS12 mask at native resolution is included in the same dataset part as the bathymetry, but still one need to compute the mask at regridded dataset resolution.

Given a vertical discretization of the domain, if one defines a valid grid cell as one filled with water by more than some given fraction of its volume, then the mask can be computed from bathymetry. 

In this way, one can also re-compute the mask at native resolution and compare that to the one provided by Mercator. 

However, we found that there is a mismatch between the the two masks for all values of the threshold. The best results are for `threshold=0.5`.

**Notice:** We weren't able to find the relevant piece of documentation saying exactly how to interpret the values of the depth coordinate. We assume that `depth` contains the depths of the cell centroids.

In [ ]:
# It is reasonable to set threshold at 0.5, that is such that a grid cell is valid if the bathymetry at its coordinates 
# is shallower than the midpoint between the bottom and top surface of the cell.
sea_mask = get_sea_mask(bathymetry, depth_dim="depth", threshold=0.5)
(bathymetry["mask"] ^ sea_mask["sea_land_mask"]).sum()

In [ ]:
# The GLORYS12 mask is stricter than (included in) the computed one
(bathymetry["mask"] & ~sea_mask["sea_land_mask"]).sum()

In [ ]:
# However, the two masks agree on surface (depth level = 0)
(bathymetry["mask"].isel(depth=0) ^ sea_mask["sea_land_mask"].isel(depth=0)).any()

Notice, that there are mismatches both between the valid values of variables from GLORYS12 and its mask, and between valid values and the computed mask.

In [ ]:
temperature = xr.open_dataset(
    "cm://cmems_mod_glo_phy_my_0.083deg_P1D-m", 
    engine="copernicusmarine", 
    variables=["thetao"], 
    start_datetime="2001-01-01",
    end_datetime="2001-01-01",
    chunks = dict(time = 1, depth = 1, latitude = -1, longitude = -1), 
)
temperature = temperature.sel(latitude=slice(-76.9167, 89.9167))
temperature = temperature.compute()
temperature

In [ ]:
(temperature["thetao"].notnull() ^ bathymetry["mask"]).sum().compute()

In [ ]:
(temperature["thetao"].notnull() ^ sea_mask["sea_land_mask"]).sum().compute()

In both cases these mismatches are all unmasked grid cells with invalid variable values, or equivalently, all not-null values that are unmasked.

In [ ]:
(temperature["thetao"].notnull() & ~bathymetry["mask"]).sum().compute()

In [ ]:
(temperature["thetao"].notnull() & ~sea_mask["sea_land_mask"]).sum().compute()

We'll see how to deal with these cells in next sections (i.e., how to extrapolate values).

Before computing the mask from regridded bathymetry, notice that selecting levels and computing the mask are in general **non commutative** operations. In this case, it happens they give identical results.

In [ ]:
sea_mask_0p25 = get_sea_mask(bathymetry_0p25, depth_dim="depth")
sea_mask_0p25 = sea_mask_0p25.isel(depth=[0, 4, 8, 12, 16, 20, 24, 28, 32, 34])
sea_mask_0p25

In [ ]:
sea_mask_0p25_after_sel = get_sea_mask(bathymetry_0p25.isel(depth=[0, 4, 8, 12, 16, 20, 24, 28, 32, 34]), depth_dim="depth")
xr.testing.assert_identical(sea_mask_0p25_after_sel["sea_land_mask"], sea_mask_0p25["sea_land_mask"])

In [ ]:
sea_mask_0p25["sea_land_mask"].hvplot.image(geo=True)

## Regridding fields

In general, when going from coarser to finer grids, interpolation algorithms does not conserve divergences (i.e., regridded solenoidal fields are generally **not** solenoidal) or other quantities of interest (e.g, energy or mass).

When going from finer to coarser grids, one encounters similar problem, although the conceptual framework is different.

For fields that represent a area densities or fluxes, the conservative algorithm will preserve total mass. However, it is customary to use it also for other fields: the idea is that values associated with larger grid cells have higher statistical significance.

When the values defined on the finer grid represent spatial averages of continuous fields over each fine grid cell, conservative algorithms will keep this property for the regridded values on the coarser grid. Hence, if one as a uniformly distributed sample of observations within a cell, he can compare their average with the previous.

Notice that the difference between computing an area weighted average using `method=conservative` and a simple average with `method=stat` is rather small, especially at low latitudes and when coarsening to a low degree.

In [ ]:
temperature_0p25 = regrid_0p25(temperature, method="conservative", skipna=True, nan_threshold=1.0)
temperature_0p25 = temperature_0p25.isel(depth=[0, 4, 8, 12, 16, 20, 24, 28, 32, 34])
temperature_0p25.hvplot.image(geo=True)

In [ ]:
temperature_0p25_stat = regrid_0p25(temperature, method="stat", reduce="mean", skipna=True, fill_value=np.nan)
(temperature_0p25_stat - temperature_0p25).hvplot.image(geo=True)

Fixed `method=conservative`, two questions still need investigation:

  1. Which value of `nan_threshold` should be used?
  2. How to deal with mismatches between variables and the sea mask computed above?


## The choice of `nan_threshold`

As commented in the section on bathymetry, when using higher values of `nan_threshold` the allowed area of the invalid finer grid cells overlapping with coarser grid cells grows.

If we imagine to compute the values of a variable on the coarser grid from uniformly distributed observations, then we would get an invalid value (a nan, i.e., a measurement on a land point) with probability at most `nan_threshold` and a valid value with probability at least `1 - nan_threshold`.  The higher `nan_threshold`, the lower the number of observations. At the same time, the mean of valid observations will be equal to the coarse grid value computed using the conservative method, no matter what `nan_threshold` is used.


Therefore, `nan_threshold=1.0` should be used, and comparing the result with `nan_threshold=0.0` should reveal more grid cells with values defined near the coastlines. 

In [ ]:
temperature_0p25_0 = regrid_0p25(temperature, method="conservative", skipna=True, nan_threshold=0.0)
temperature_0p25_0 = temperature_0p25_0.isel(depth=[0, 4, 8, 12, 16, 20, 24, 28, 32, 34])
(temperature_0p25 - temperature_0p25_0.where(temperature_0p25_0.notnull(), 0.0)).hvplot.image(geo=True)

## Comparison between regridded fields and sea mask


Suppose that the set of the finer grid cells containing a valid value of some variable is contained within the set of unmasked cells (with respect to the sea mask computed from bathymetry). Then, if a coarse grid cell contains a valid value of the regridded field with `nan_threshold=0.0`, then that cell has zero area overlap with finer grid cells containing invalid values. But then, that coarse grid cell overlaps only with unmasked cells. Therefore, when computing the mask from the regridded bathymetry using the same depth levels, that grid cell will be filled with water by more than half of its volume[^1]. 

In other terms, when all not null values are unmasked on on the finer grid (as in GLORYS12) and `nan_threshold=0.0`, then all not null values will be unmasked on the coarser grid.

In general, regridded fields computed using `nan_threshold=1.0` could have valid values outside the mask, which however could be masked afterwards.


[^1]: Each overlapping finer grid cell contributes with an amount of water greater than half of its overlapping volume (as they are all valid grid cells), hence the total volume of water is more than half of the total overlapping volume.

In [ ]:
# A cell with valid temperature is unmasked when nan_threshold = 0.0
(temperature_0p25_0["thetao"].notnull() & ~sea_mask_0p25["sea_land_mask"]).sum()

In [ ]:
# This is no more the case for nan_threshold = 1.0
(temperature_0p25["thetao"].notnull() & ~sea_mask_0p25["sea_land_mask"]).sum()

In [ ]:
# Unfortunately, not all null/unmasked points are lost in the regridding process when nan_threshold=1.0
(temperature_0p25["thetao"].isnull() & sea_mask_0p25["sea_land_mask"]).sum().compute()

In [ ]:
(temperature_0p25["thetao"].isnull() & sea_mask_0p25["sea_land_mask"]).hvplot.image(geo=True)

In [ ]:
# And it is worse in the case when nan_threshold=0.0
(temperature_0p25_0["thetao"].isnull() & sea_mask_0p25["sea_land_mask"]).sum().compute()

In cases when the problem of null values defined on the unmasked grid cells persists after regridding, one needs to extrapolate regridded fields.

We can extrapolate the values of the regridded field near the coastlines using weighted sums of nearby points. An implementation in pure Python is not feasible, and we rely on a faster implementation of [gaussian filtering available in SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html). The implementation takes care of invalid values by filling with zeros beforehand, and rescaling the result afterward (see implementation for details).

Using a gaussian kernel has the advantage of weighting more closer grid cells. The optimal values of kernel parameters, such as its size and $\sigma$, have to be determined by experiments.

In [ ]:
temperature_0p25_fix = gaussian_blur_extrapolate(temperature_0p25.copy(deep=True), gaussian_filter_kwargs = dict(sigma = 1.0, radius = 5, mode = "mirror"))
(temperature_0p25_fix["thetao"].isnull() & sea_mask_0p25["sea_land_mask"]).sum().compute()

In [ ]:
temperature_0p25_fix.where(temperature_0p25["thetao"].isnull() & sea_mask_0p25["sea_land_mask"], drop=True).compute()

In [ ]:
temperature_0p25_fix.where(sea_mask_0p25["sea_land_mask"]).hvplot.image(geo=True)